# 🌍 Universal Crypto Transformer Training (Top 100 Symbols)

This notebook trains a single **"Generalist" Transformer Model** capable of trading any liquid crypto asset.
Instead of learning just Bitcoin's patterns, it learns the universal language of market dynamics by studying the top 100 coins simultaneously.

### 🚀 Key Features:
1.  **Dynamic Asset Selection**: Automatically fetches the top 100 coins by volume from Bybit.
2.  **Massive Dataset**: Downloads ~180 days of 5-minute data for ALL 100 coins.
3.  **Universal Normalization**: Uses percentage-based features (RSI, MACD, Returns) so the model works on any price scale (from PEPE's $0.00001 to BTC's $60,000).
4.  **Robust Training**: Uses the same Adversarial Training & Volatility-Weighted Loss as the BTC model.

In [ ]:
# 1. Setup & Dependencies
!pip install -q pybit pandas numpy torch matplotlib scikit-learn

import os
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from pybit.unified_trading import HTTP
import matplotlib.pyplot as plt
from datetime import datetime
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 2. Feature Engineering (Universal)
# Note: We use the exact same 12 features as the BTC model to ensure compatibility.

def prepare_features(df):
    df = df.copy()
    
    # --- 1. Trend & Momentum ---
    df['returns'] = df['close'].pct_change()
    df['log_ret'] = np.log(df['close'] / df['close'].shift(1))
    df['mom_1h'] = df['close'].pct_change(12) # 1 Hour momentum (5m * 12)
    df['mom_4h'] = df['close'].pct_change(48) # 4 Hour momentum
    
    # --- 2. Volatility ---
    df['volatility'] = df['log_ret'].rolling(20).std()
    df['atr'] = (np.maximum(df['high'] - df['low'], 
                 np.maximum(abs(df['high'] - df['close'].shift(1)), 
                            abs(df['low'] - df['close'].shift(1))))).rolling(14).mean()
    df['atr_pct'] = df['atr'] / df['close']
    
    # --- 3. Oscillators ---
    # RSI
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # MACD
    ema12 = df['close'].ewm(span=12).mean()
    ema26 = df['close'].ewm(span=26).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9).mean()
    df['macd_hist'] = macd - signal
    
    # --- 4. Relative Position ---
    # Bollinger Bands
    sma20 = df['close'].rolling(20).mean()
    std20 = df['close'].rolling(20).std()
    df['bb_width'] = (4 * std20) / sma20
    df['bb_pos'] = (df['close'] - (sma20 - 2*std20)) / (4*std20)
    
    # Distance from EMAs
    ema50 = df['close'].ewm(span=50).mean()
    df['dist_ema50'] = (df['close'] / ema50) - 1
    
    # --- 5. Volume ---
    vol_ma20 = df['vol'].rolling(20).mean()
    df['vol_ratio'] = df['vol'] / (vol_ma20 + 1e-8)
    
    # --- Targets ---
    horizon = 12 # 1 Hour
    df['future_close'] = df['close'].shift(-horizon)
    df['future_ret'] = (df['future_close'] - df['close']) / df['close']
    
    # Adaptive Labeling (0.6% threshold)
    threshold = 0.006
    conditions = [
        (df['future_ret'] < -threshold), # Short
        (df['future_ret'] > threshold)   # Long
    ]
    df['target'] = np.select(conditions, [0, 2], default=1)
    
    # Cleanup
    df = df.dropna()
    
    # Feature Selection (12 Features)
    features = ['log_ret', 'volatility', 'rsi', 'macd_hist', 'bb_width', 'bb_pos', 
                'atr_pct', 'vol_ratio', 'dist_ema50', 'mom_1h', 'mom_4h', 'returns']
                
    # Normalize Features (Global Z-Score)
    for col in features:
        df[col] = (df[col] - df[col].mean()) / df[col].std()
        
    return df, features

In [ ]:
# 3. Transformer Model Architecture (Same as BTC Model)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class BTCTransformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, num_layers=3, dropout=0.2):
        super().__init__()
        # Input Projection
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, 
                                                 dim_feedforward=d_model*4, 
                                                 dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Output Heads
        self.direction_head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 3) # [Short, Neutral, Long]
        )
        
    def forward(self, x):
        # x: [batch, seq_len, features]
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        
        # Transformer Pass
        x = self.transformer_encoder(x)
        
        # Global Average Pooling (Focus on the whole sequence context)
        x = x.mean(dim=1)
        
        return self.direction_head(x)

In [ ]:
# 4. Robust Loss & Augmentation
class DataAugmenter:
    def __init__(self, noise_level=0.02, mask_prob=0.1):
        self.noise_level = noise_level
        self.mask_prob = mask_prob

    def augment(self, x: torch.Tensor) -> torch.Tensor:
        x_aug = x.clone()
        if np.random.random() < 0.5:
            noise = torch.randn_like(x_aug) * self.noise_level
            x_aug += noise
        if np.random.random() < 0.3:
            mask = torch.bernoulli(torch.full_like(x_aug, 1 - self.mask_prob))
            x_aug *= mask
        return x_aug

class RobustLoss(nn.Module):
    def __init__(self, volatility_penalty=2.0, smoothing=0.1):
        super().__init__()
        self.vol_penalty = volatility_penalty
        self.smoothing = smoothing

    def forward(self, predictions, targets, volatility=None):
        n_classes = predictions.size(1)
        one_hot = torch.zeros_like(predictions).scatter(1, targets.view(-1, 1), 1)
        one_hot = one_hot * (1 - self.smoothing) + (1 - one_hot) * self.smoothing / (n_classes - 1)
        log_probs = F.log_softmax(predictions, dim=1)
        loss = -(one_hot * log_probs).sum(dim=1)
        if volatility is not None:
            weights = 1.0 + F.relu(volatility * self.vol_penalty)
            loss = loss * weights
        return loss.mean()

In [ ]:
# 5. Dynamic Data Fetching (Top 100 Symbols)
session = HTTP()

def get_top_100_symbols():
    print("🔍 Scanning for Top 100 Liquid Symbols...")
    try:
        resp = session.get_tickers(category="linear")
        if resp['retCode'] != 0: return []
        
        items = resp['result']['list']
        scored = []
        for it in items:
            if not it['symbol'].endswith('USDT'): continue
            turnover = float(it.get('turnover24h', 0))
            scored.append((turnover, it['symbol']))
        
        scored.sort(reverse=True)
        top_100 = [s for _, s in scored[:100]]
        print(f"✅ Found {len(top_100)} symbols: {top_100[:5]} ...")
        return top_100
    except Exception as e:
        print(f"❌ Error fetching symbols: {e}")
        return []

def fetch_history_single(symbol, days=180):
    # print(f"   Fetching {symbol}...")
    end_time = int(time.time() * 1000)
    all_klines = []
    limit = 1000
    interval = "5"
    total_candles = (days * 24 * 60) // 5
    
    # Only fetch last 30 days per symbol to save time/memory for 100 symbols
    # 30 days * 100 symbols = 3000 days of data (Massive!)
    days_per_symbol = 60 
    total_candles = (days_per_symbol * 24 * 60) // 5
    
    while len(all_klines) < total_candles:
        try:
            resp = session.get_kline(category="linear", symbol=symbol, interval=interval, limit=limit, end=end_time)
            if not resp['result']['list']:
                break
            batch = resp['result']['list']
            all_klines.extend(batch)
            end_time = int(batch[-1][0]) - 1
            time.sleep(0.05) # Rate limit protection
        except Exception:
            break
            
    if len(all_klines) == 0: return None
    
    df = pd.DataFrame(all_klines, columns=['ts', 'open', 'high', 'low', 'close', 'vol', 'turn'])
    df = df.astype(float)
    df = df.sort_values('ts').reset_index(drop=True)
    return df

def build_universal_dataset():
    symbols = get_top_100_symbols()
    all_dataframes = []
    
    print(f"🚀 Starting Mass Download for {len(symbols)} symbols...")
    
    # Sequential download to avoid rate limits
    for i, sym in enumerate(symbols):
        print(f"[{i+1}/{len(symbols)}] Downloading {sym}...", end="\r")
        df = fetch_history_single(sym)
        if df is not None and len(df) > 1000:
            processed, _ = prepare_features(df)
            all_dataframes.append(processed)
            
    print(f"\n✅ Successfully processed {len(all_dataframes)} symbols.")
    
    # Combine all data
    if not all_dataframes: return None, None
    
    universal_df = pd.concat(all_dataframes, ignore_index=True)
    print(f"📊 Total Training Samples: {len(universal_df):,}")
    print("Class Distribution:", universal_df['target'].value_counts(normalize=True))
    
    return universal_df, ['log_ret', 'volatility', 'rsi', 'macd_hist', 'bb_width', 'bb_pos', 
                          'atr_pct', 'vol_ratio', 'dist_ema50', 'mom_1h', 'mom_4h', 'returns']

universal_df, feature_cols = build_universal_dataset()

In [ ]:
# 6. Dataset & Loader
class RobustDataset(Dataset):
    def __init__(self, data, targets, volatility, seq_len=72, augment=False):
        self.data = torch.FloatTensor(data)
        self.targets = torch.LongTensor(targets)
        self.volatility = torch.FloatTensor(volatility)
        self.seq_len = seq_len
        self.augment = augment
        self.augmenter = DataAugmenter()

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.seq_len]
        y = self.targets[idx+self.seq_len-1]
        vol = self.volatility[idx+self.seq_len-1]
        if self.augment:
            x = self.augmenter.augment(x)
        return x, y, vol

# Split Data
train_size = int(len(universal_df) * 0.8)
train_df = universal_df.iloc[:train_size]
val_df = universal_df.iloc[train_size:]

train_dataset = RobustDataset(train_df[feature_cols].values, train_df['target'].values, train_df['volatility'].values, augment=True)
val_dataset = RobustDataset(val_df[feature_cols].values, val_df['target'].values, val_df['volatility'].values, augment=False)

# Weighted Sampler
class_counts = np.bincount(train_df['target'])
class_weights = 1. / class_counts
sample_weights = class_weights[train_df['target'].values[72:]]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

# Larger Batch Size for Massive Dataset
train_loader = DataLoader(train_dataset, batch_size=256, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=256)

In [ ]:
# 7. Training Loop
model = BTCTransformer(input_dim=len(feature_cols)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
criterion = RobustLoss(volatility_penalty=3.0)

epochs = 30 # Fewer epochs needed for massive data
best_val_loss = float('inf')

print("🔥 Starting Universal Transformer Training...")
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for x, y, vol in train_loader:
        x, y, vol = x.to(device), y.to(device), vol.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y, vol)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y, vol in val_loader:
            x, y, vol = x.to(device), y.to(device), vol.to(device)
            pred = model(x)
            loss = criterion(pred, y, vol)
            val_loss += loss.item()
            _, predicted = torch.max(pred.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()
            
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    acc = 100 * correct / total
    
    print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, Val Acc={acc:.2f}%")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "universal_transformer_best.pt")
        print("   ✅ Saved Best Model")

In [ ]:
# 8. Download Model
from google.colab import files
files.download('universal_transformer_best.pt')